# 09 - Comparacion: dataset real vs sintetico

**Audiencia:** docente y equipo, para decidir el uso del dataset sintetico.

Este notebook compara `data/clean/dashboard_real` con
`data/synthetic/dashboard_sintetico` (ambos generados por el notebook 08, con
esquema identico de 39 columnas) con dos objetivos:

- **(a)** demostrar que el sintetico es **estadisticamente plausible** respecto
  al real: mismas proporciones por facultad/instrumento/anio y distribuciones
  numericas de la misma forma;
- **(b)** mostrar **que metricas puede alimentar cada uno**: el real tiene
  columnas estructuralmente vacias (datos que la fuente no captura), el
  sintetico las tiene todas pobladas.

Convenciones visuales de todo el notebook:

- **Azul = dataset real, verde aqua = dataset sintetico**, en todos los graficos
  (el color sigue al dataset, nunca cambia de rol).
- Todo grafico que use datos sinteticos lleva la etiqueta
  **[DATOS SINTETICOS / DEMOSTRATIVOS]**.
- El sintetico es solo para demostracion del dashboard: ninguna de sus filas
  corresponde a una iniciativa real.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

CLEAN = Path("..") / "data" / "clean"
SYNTH = Path("..") / "data" / "synthetic"

real = pd.read_parquet(CLEAN / "dashboard_real.parquet")
sint = pd.read_parquet(SYNTH / "dashboard_sintetico.parquet")

# mismo esquema, mismo orden: si no, este notebook no tiene sentido
assert list(real.columns) == list(sint.columns), "Esquemas distintos entre real y sintetico"
assert len(real) == len(sint)
print(f"Esquema identico confirmado: {len(real.columns)} columnas, {len(real)} filas cada uno")

# paleta (referencia validada): el color sigue al dataset en todo el notebook
C_REAL = "#2a78d6"   # azul, slot categorico 1
C_SINT = "#1baf7a"   # aqua, slot categorico 2
C_TER = "#eda100"    # amarillo, slot 3 (tercer segmento en apilados)
INK = "#0b0b0b"
MUTED = "#898781"
GRID = "#e1e0d9"
SEQ_AZUL = LinearSegmentedColormap.from_list("seq_azul", [
    "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])

ETIQUETA_SINT = "[DATOS SINTETICOS / DEMOSTRATIVOS]"


def estilo(ax, titulo=None):
    """Chrome recesivo: sin bordes, grilla fina, tinta para texto."""
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#c3c2b7")
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, labelsize=9)
    if titulo:
        ax.set_title(titulo, color=INK, fontsize=11, loc="left")


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

## Seccion 1 - Cobertura de metricas comparada

Que porcentaje de las 826 filas tiene dato, en cada dataset, para cada
metrica/columna que pidio la contraparte. Es la foto de **que graficos del
dashboard puede alimentar cada dataset**: donde el real marca 0%, ese dato no
existe en la fuente y solo el sintetico puede demostrar el grafico.

Nota de lectura: en el real, los participantes desagregados rondan 60% del total
de filas porque solo existen desde 2024 (las filas 2022-2023 estan vacias por
diseno, no por error).

In [ ]:
METRICAS = [
    ("n_estudiantes", "Estudiantes"),
    ("n_academicos", "Academicos / docentes"),
    ("n_titulados", "Titulados"),
    ("n_empleadores", "Empleadores"),
    ("n_organizaciones_osc", "Organizaciones OSC"),
    ("n_titulados_charlista", "Roles: titulados charlista"),
    ("n_titulados_expositor", "Roles: titulados expositor"),
    ("n_titulados_asistente", "Roles: titulados asistente"),
    ("n_empleadores_charlista", "Roles: empleadores charlista"),
    ("n_empleadores_expositor", "Roles: empleadores expositor"),
    ("n_empleadores_asistente", "Roles: empleadores asistente"),
    ("n_organizaciones_osc_charlista", "Roles: organizaciones charlista"),
    ("n_organizaciones_osc_expositor", "Roles: organizaciones expositor"),
    ("n_organizaciones_osc_asistente", "Roles: organizaciones asistente"),
    ("competencia_sello", "Competencia sello"),
    ("competencia_generica", "Competencia generica"),
    ("dominios_disciplinares", "Dominios disciplinares"),
    ("ciclo_estudio", "Ciclo de estudio"),
    ("internacionalizacion", "Internacionalizacion"),
    ("comuna_rm", "Comuna RM"),
    ("evidencia", "Evidencia de ejecucion"),
    ("cantidad_act_planificadas", "KPI I19: act. planificadas"),
    ("cantidad_act_ejecutadas", "KPI I19: act. ejecutadas"),
]

cob = pd.DataFrame({
    "Real (%)": [round(100 * real[c].notna().mean(), 1) for c, _ in METRICAS],
    "Sintetico (%)": [round(100 * sint[c].notna().mean(), 1) for c, _ in METRICAS],
}, index=[nombre for _, nombre in METRICAS])

print(cob.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8.5))
sns.heatmap(cob, annot=True, fmt=".0f", cmap=SEQ_AZUL, vmin=0, vmax=100,
            linewidths=2, linecolor="white", cbar_kws={"label": "% de filas con dato"},
            ax=ax)
ax.set_title("Cobertura por metrica: real vs sintetico\n(sintetico: datos demostrativos)",
             color=INK, fontsize=11, loc="left")
ax.tick_params(colors=MUTED, labelsize=9)
plt.tight_layout()
plt.show()

**Lectura.** El patron es exactamente el esperado:

- El **real** solo puebla lo que la fuente captura: participantes desagregados,
  competencia generica y dominios desde 2024 (por eso ~60% del total),
  competencia sello, evidencia y actividades planificadas en casi todo el
  periodo, actividades ejecutadas solo 2022-2023 (~31% del total), y **0% en
  empleadores, roles, ciclo, internacionalizacion y comuna**, porque el
  instrumento no registra esos datos.
- El **sintetico** esta al 100% en todas las metricas.

Todo grafico del dashboard que dependa de las filas en 0% del real **solo puede
demostrarse con el sintetico** (seccion 3). El resto puede construirse con datos
reales verificados.

## Seccion 2 - Similitud de distribuciones

Aca se compara **solo lo que existe en ambos con datos reales**: los conteos de
participantes (contra las filas reales 2024-2025, que son las que tienen dato) y
la composicion por facultad, instrumento y anio.

In [ ]:
NUMERICAS = [
    ("n_estudiantes", "N estudiantes"),
    ("n_academicos", "N academicos"),
    ("n_titulados", "N titulados"),
    ("n_organizaciones_osc", "N organizaciones OSC"),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (col, nombre) in zip(axes.ravel(), NUMERICAS):
    vr = real[col].dropna()
    vs = sint[col].dropna()
    tope = np.percentile(pd.concat([vr, vs]), 99)  # recorte visual del 1% extremo
    bins = np.histogram_bin_edges(np.clip(pd.concat([vr, vs]), 0, tope), bins=30)
    # real relleno, sintetico como contorno: superpuestos sin fundir colores
    ax.hist(np.clip(vr, 0, tope), bins=bins, density=True, alpha=0.85,
            color=C_REAL, label="Real", edgecolor="white", linewidth=0.4)
    ax.hist(np.clip(vs, 0, tope), bins=bins, density=True, histtype="step",
            color=C_SINT, label="Sintetico", linewidth=2.2)
    estilo(ax, nombre)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK)
fig.suptitle("Distribucion de conteos: real (2024-2025) vs sintetico  " + ETIQUETA_SINT,
             color=INK, fontsize=12, x=0.02, ha="left")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

**Lectura.** Las cuatro distribuciones sinteticas tienen la misma forma que
las reales: fuerte concentracion cerca de cero con cola larga (pocas iniciativas
masivas), medianas identicas y el mismo rango. Es lo esperable porque el
sintetico se remuestrea de la distribucion empirica real. El eje se recorta en el
percentil 99 solo para que la forma sea visible; los extremos existen en ambos.

In [ ]:
DIMENSIONES = [("facultad", "Por facultad"), ("instrumento", "Por instrumento"),
               ("anio", "Por anio")]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (col, nombre) in zip(axes, DIMENSIONES):
    pr = real[col].value_counts(normalize=True).sort_index()
    ps = sint[col].value_counts(normalize=True).sort_index()
    cats = pr.index.astype(str).tolist()
    x = np.arange(len(cats))
    ax.bar(x - 0.2, [pr.get(c, 0) if col != "anio" else pr.iloc[i] for i, c in enumerate(pr.index)],
           width=0.38, color=C_REAL, label="Real", edgecolor="white", linewidth=0.5)
    ax.bar(x + 0.2, [ps.reindex(pr.index).fillna(0).iloc[i] for i in range(len(pr))],
           width=0.38, color=C_SINT, label="Sintetico", edgecolor="white", linewidth=0.5)
    ax.set_xticks(x, cats, rotation=45, ha="right")
    estilo(ax, nombre)
axes[0].legend(frameon=False, fontsize=9, labelcolor=INK)
axes[0].set_ylabel("Proporcion de iniciativas", color=MUTED, fontsize=9)
fig.suptitle("Composicion de iniciativas: real vs sintetico  " + ETIQUETA_SINT,
             color=INK, fontsize=12, x=0.02, ha="left")
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

**Lectura.** Las proporciones por facultad e instrumento del sintetico siguen
de cerca a las reales (diferencias de 1 a 3 puntos porcentuales, propias del
remuestreo). La distribucion **por anio es identica por construccion**: el
generador produce exactamente las mismas filas por anio que el real, con 2025
como el anio de mayor volumen.

In [ ]:
filas = []
for col, nombre in NUMERICAS:
    vr, vs = real[col].dropna(), sint[col].dropna()
    dm = 100 * abs(vs.mean() - vr.mean()) / vr.mean() if vr.mean() else np.nan
    ds = 100 * abs(vs.std() - vr.std()) / vr.std() if vr.std() else np.nan
    peor = np.nanmax([dm, ds])
    clas = "BAJA (buena similitud)" if peor <= 15 else (
        "MEDIA" if peor <= 30 else "ALTA (revisar)")
    filas.append({
        "metrica": nombre,
        "media real": round(vr.mean(), 1), "media sint": round(vs.mean(), 1),
        "mediana real": vr.median(), "mediana sint": vs.median(),
        "desv real": round(vr.std(), 1), "desv sint": round(vs.std(), 1),
        "dif media %": round(dm, 1), "dif desv %": round(ds, 1),
        "diferencia": clas,
    })
resumen = pd.DataFrame(filas).set_index("metrica")
print(resumen.to_string())

**Lectura.** Las medias y medianas quedan practicamente calcadas (diferencias
bajo el 10%). La desviacion de los conteos mas dispersos (titulados) puede
diferir mas, porque la cola extrema de esa variable depende de pocas iniciativas
gigantes y el remuestreo puede tomar mas o menos de ellas; la forma general se
mantiene (ver histogramas). Para fines de demostracion del dashboard, la
similitud es suficiente: los graficos agregados del sintetico se ven como se
verian con datos reales.

## Seccion 3 - Metricas del dashboard demostradas con el sintetico

Estos son ejemplos de los graficos que la contraparte pidio y que **solo el
sintetico puede alimentar hoy** (en el real esas columnas estan vacias porque la
fuente no captura el dato). Muestran como se veria el dashboard con captura
completa. **Todos usan datos sinteticos / demostrativos.**

In [ ]:
# Demo 1: participantes desagregados por rol
cats = [("n_titulados", "Titulados"), ("n_empleadores", "Empleadores"),
        ("n_organizaciones_osc", "Organizaciones OSC")]
roles = ["charlista", "expositor", "asistente"]
colores_rol = [C_REAL, C_SINT, C_TER]

fig, ax = plt.subplots(figsize=(9, 3.6))
y = np.arange(len(cats))
izq = np.zeros(len(cats))
for rol, color in zip(roles, colores_rol):
    vals = np.array([sint[f"{c}_{rol}"].sum() for c, _ in cats])
    ax.barh(y, vals, left=izq, height=0.55, color=color,
            edgecolor="white", linewidth=2, label=rol.capitalize())
    izq += vals
ax.set_yticks(y, [n for _, n in cats])
ax.invert_yaxis()
ax.grid(axis="x", color=GRID, linewidth=0.8)
ax.grid(axis="y", visible=False)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.tick_params(colors=MUTED, labelsize=9)
ax.set_title("Participantes por rol (suma 2022-2025)  " + ETIQUETA_SINT,
             color=INK, fontsize=11, loc="left")
# leyenda fuera del area de datos para no pisar la primera barra
ax.legend(frameon=False, fontsize=9, labelcolor=INK, ncols=3,
          loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.show()

In [ ]:
# Demo 2: iniciativas por ciclo de estudio + top comunas RM
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

ciclo = sint["ciclo_estudio"].value_counts().sort_index()
ax1.bar(ciclo.index.astype(str), ciclo.values, width=0.55, color=C_SINT,
        edgecolor="white", linewidth=0.5)
for i, v in enumerate(ciclo.values):
    ax1.text(i, v + 5, str(v), ha="center", color=MUTED, fontsize=9)
estilo(ax1, "Iniciativas por ciclo de estudio  " + ETIQUETA_SINT)
ax1.set_xlabel("Ciclo", color=MUTED, fontsize=9)

top = sint["comuna_rm"].value_counts().head(12).sort_values()
ax2.barh(top.index, top.values, height=0.6, color=C_SINT,
         edgecolor="white", linewidth=0.5)
ax2.grid(axis="x", color=GRID, linewidth=0.8)
ax2.grid(axis="y", visible=False)
for s in ("top", "right", "left"):
    ax2.spines[s].set_visible(False)
ax2.tick_params(colors=MUTED, labelsize=9)
ax2.set_title("Top 12 comunas RM por iniciativas  " + ETIQUETA_SINT,
              color=INK, fontsize=11, loc="left")
plt.tight_layout()
plt.show()

In [ ]:
# Demo 3: KPI I19, actividades planificadas vs ejecutadas por anio
kpi = sint.groupby("anio")[["cantidad_act_planificadas", "cantidad_act_ejecutadas"]].sum()
cumpl = 100 * kpi["cantidad_act_ejecutadas"] / kpi["cantidad_act_planificadas"]

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(kpi))
ax.bar(x - 0.2, kpi["cantidad_act_planificadas"], width=0.38, color=C_REAL,
       edgecolor="white", linewidth=0.5, label="Planificadas")
ax.bar(x + 0.2, kpi["cantidad_act_ejecutadas"], width=0.38, color=C_SINT,
       edgecolor="white", linewidth=0.5, label="Ejecutadas")
for i, (p, e, c) in enumerate(zip(kpi.iloc[:, 0], kpi.iloc[:, 1], cumpl)):
    ax.text(i + 0.2, e + 8, f"{c:.0f}%", ha="center", color=INK, fontsize=9)
ax.set_xticks(x, kpi.index.astype(str))
estilo(ax, "KPI I19: actividades planificadas vs ejecutadas (% cumplimiento)  " + ETIQUETA_SINT)
ax.legend(frameon=False, fontsize=9, labelcolor=INK)
plt.tight_layout()
plt.show()

**Lectura de la seccion.** Con captura completa, el dashboard podria mostrar:
el reparto de cada categoria de participante por rol (asistentes dominan, como
se parametrizo), la carga por ciclo de estudio, la huella territorial por comuna
de la RM, y el cumplimiento del KPI I19 (~85% por construccion, coherente con la
regla `ejecutadas <= planificadas`). **Ninguna de estas cifras es real**: son el
molde visual del dashboard, no un resultado institucional.

## Seccion 4 - Conclusion y recomendacion de uso

**Fidelidad del sintetico (seccion 2).** El sintetico reproduce las proporciones
reales por facultad, instrumento y anio (diferencias de pocos puntos), y las
distribuciones numericas de participantes con medias y medianas practicamente
identicas; solo la dispersion de las variables de cola larga (titulados) se
desvia algo mas, sin cambiar la forma. Para efectos visuales y de demostracion,
es indistinguible del real en agregado.

**Que habilita cada dataset (secciones 1 y 3).**

| Metrica | Real | Sintetico |
|---|---|---|
| Iniciativas, facultad, carrera, instrumento, anio, estado | Si (verificado) | Si (demo) |
| Participantes desagregados (estudiantes, academicos, titulados, organizaciones) | Si, solo 2024-2025 | Si (demo) |
| Competencia sello (y flags) | Si (2022-2025) | Si (demo) |
| Competencia generica | Si, solo 2024-2025 | Si (demo) |
| Catedra / asignatura | Parcial (~57% en 2024-2025) | Si (demo) |
| Evidencia de ejecucion (SI/NO) | Si (2022-2025, normalizada) | Si (demo) |
| KPI I19 (ejecutadas/planificadas) | Si, solo 2022-2023 | Si (demo) |
| Empleadores, roles (charlista/expositor/asistente) | No (la fuente no lo captura) | Solo demo |
| Ciclo, internacionalizacion, comuna RM | No (la fuente no lo captura) | Solo demo |

**Recomendacion.**

- **Sintetico**: para demostrar el dashboard completo a la contraparte (todas las
  vistas funcionando), etiquetado siempre como demostrativo. Nunca para
  reporteria, informes ni acreditacion.
- **Real**: para todo lo verificado que existe (iniciativas, participantes
  2024-2025, competencias); es la unica fuente valida para cifras
  institucionales.
- **La linea entre ambos** esta trazada en los datos mismos: la columna
  `_origen_dato` ("real" / "sintetico") permite que el dashboard muestre la
  procedencia en cada vista, y las columnas vacias del real son la hoja de ruta
  de que deberia empezar a capturar VcM para reemplazar cada grafico demo por su
  version real.